# Evaluation of ASR-based subtitling in Spanish

This notebook computes all the metrics and statistical analyses of the paper
*A Multidimensional Evaluation Framework for Accessibility-Oriented Automatic Subtitling in ASR Systems*.

**Two ways of running it**

1. **Full recomputation** (requires the ASR outputs and the reference transcriptions, available from the
   authors on request). Place the reference files in `data/references/` (`TEXTO-REFERENCIA-VIDEO-N.txt`) and the
   outputs of `scripts/rerun_controlled.py` and `scripts/punctuation_restoration_wav2vec.py` in
   `data/asr_outputs/VIDEO-N/`. A GPU is recommended for the semantic metrics.
2. **Statistics only** (no data needed). If `data/asr_outputs/` does not exist, the notebook loads the
   per-video results of `results/` and recomputes the statistical analysis, tables and figures.

All new files are written to `results_rerun/`, so they can be compared with the frozen files in `results/`.

| Section | Output (in `results_rerun/`) | Paper |
|---|---|---|
| 4. Per-video metrics | `results_per_video_raw.csv` | Tables 4, 5 and 7, Figs. 3-4 |
| 5. Subtitle metrics | `subtitle_metrics.csv` | Table 6, Fig. 5 |
| 6. Ablation | `ablation_postprocessing.csv`, `ablation_segmentation.csv` | Ablation table |
| 7. Punctuation restoration (Wav2Vec2) | `punctuation_restoration_*.csv` | Punctuation restoration table |
| 8. Statistics | `stats/friedman.csv`, `stats/wilcoxon_holm.csv`, `stats/descriptive.csv`, `stats/sensitivity.csv` | Tables 7-9 |

## 0. Environment

In [ ]:
# Optional (Google Colab): mount Google Drive and install the packages that are not preinstalled.
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jiwer==4.0.0", "bert-score==0.3.13",
                    "textstat==0.7.13", "sentence-transformers", "statsmodels", "seaborn"], check=True)

In [ ]:
import os, re, glob, math, time, itertools
from pathlib import Path
import numpy as np
import pandas as pd
import jiwer
import textstat
from scipy.stats import friedmanchisquare, wilcoxon, norm
from statsmodels.stats.multitest import multipletests

textstat.set_lang("es")
pd.set_option("display.max_columns", 50)

## 1. Configuration

`REPO_DIR` is detected automatically when the notebook is opened from `notebooks/`.
In Colab, set it to the folder where the repository was cloned.

In [ ]:
REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# REPO_DIR = Path("/content/drive/MyDrive/asr-subtitling-eval")      # example for Colab

REF_DIR    = REPO_DIR / "data" / "references"     # TEXTO-REFERENCIA-VIDEO-N.txt
ASR_DIR    = REPO_DIR / "data" / "asr_outputs"    # VIDEO-N/<model>__<variant>.txt and .srt
FROZEN_DIR = REPO_DIR / "results"                 # results reported in the paper
OUT_DIR    = REPO_DIR / "results_rerun"           # results of this execution
(OUT_DIR / "stats").mkdir(parents=True, exist_ok=True)

RECOMPUTE = ASR_DIR.exists() and any(ASR_DIR.glob("VIDEO-*"))
print("Mode:", "full recomputation from the ASR outputs" if RECOMPUTE else "statistics only (frozen per-video results)")

MAIN_VARIANT = "raw"   # raw ASR output, without post-processing
MODEL_ORDER = ["Tiny", "Base", "Small", "Wav2Vec2", "Medium", "Large-v1", "Large-v2", "Large-v3", "Large-v3-turbo"]
ALIAS_TO_NAME = {"tiny": "Tiny", "base": "Base", "small": "Small", "medium": "Medium",
                 "large-v1": "Large-v1", "large-v2": "Large-v2", "large-v3": "Large-v3",
                 "large-v3-turbo": "Large-v3-turbo", "wav2vec2": "Wav2Vec2"}
PARAMS = {"Wav2Vec2": "317 M", "Tiny": "39 M", "Base": "74 M", "Small": "244 M", "Medium": "769 M",
          "Large-v1": "1550 M", "Large-v2": "1550 M", "Large-v3": "1550 M", "Large-v3-turbo": "809 M"}
TERMS = ["CESyA", "CEACOG"]   # domain terms for term recall (only CEACOG appears in the references)

# Subtitle constraints derived from UNE 153010:2012
MAX_CPL, MAX_LINES, MAX_CPS, MIN_DUR, MAX_DUR = 37, 2, 15.0, 1.0, 6.0
VARIANTS = ["raw", "norm", "prompt", "prompt+norm", "punct"]   # "punct" exists only for Wav2Vec2 (Section 7)
SEGMENTATIONS = ["native", "rules", "une"]

# Statistics
N_BOOT, SEED, ALPHA = 10_000, 42, 0.05
SENSITIVITY_WER = 0.02   # videos where any model obtains WER < 0.02 are excluded in the sensitivity analysis
REF_LABEL = "Referencia (Gold Standard)"

def ref_path(n):
    c = [p for p in glob.glob(f"{REF_DIR}/*VIDEO-{n}.txt") if os.path.basename(p).upper().startswith("TEXTO")]
    return c[0] if c else None

def build_config(variant):
    # {"Video N": {"referencia": path, "hipotesis": {model: path}}} for one output variant
    cfg = {}
    for vdir in sorted(glob.glob(f"{ASR_DIR}/VIDEO-*"), key=lambda p: int(p.rsplit("-", 1)[1])):
        n = int(vdir.rsplit("-", 1)[1])
        hyp = {}
        for p in sorted(glob.glob(f"{vdir}/*__{variant}.txt")):
            alias = os.path.basename(p).split("__")[0]
            hyp[ALIAS_TO_NAME.get(alias, alias)] = p
        cfg[f"Video {n}"] = {"referencia": ref_path(n), "hipotesis": hyp}
    return cfg

if RECOMPUTE:
    CONFIG = build_config(MAIN_VARIANT)
    missing = {v: sorted(set(MODEL_ORDER) - set(d["hipotesis"])) for v, d in CONFIG.items()}
    missing = {v: m for v, m in missing.items() if m}
    assert len(CONFIG) == 15, f"{len(CONFIG)} videos found in {ASR_DIR}, 15 expected"
    assert not missing, f"Missing model outputs: {missing}"
    assert all(d["referencia"] for d in CONFIG.values()), "Missing reference transcription"
    print("OK: 15 videos x 9 models, with references")

## 2. Encoders for the semantic metrics

* **BERTScore** (F1, no baseline rescaling) and **SeMaScore** use XLM-RoBERTa large, layer 17.
* **SBERT-cos** (supplementary) uses `paraphrase-multilingual-MiniLM-L12-v2`.

In [ ]:
ENCODER_MODEL = "xlm-roberta-large"
ENCODER_LAYER = 17          # layer recommended by bert-score for xlm-roberta-large

if RECOMPUTE:
    import torch
    from bert_score import BERTScorer
    from sentence_transformers import SentenceTransformer, util as st_util
    from transformers import AutoTokenizer, AutoModel
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("Loading encoders on", DEVICE, "(the first time this downloads about 2 GB)")
    BS = BERTScorer(model_type=ENCODER_MODEL, num_layers=ENCODER_LAYER, lang="es",
                    rescale_with_baseline=False, device=DEVICE)
    SBERT = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device=DEVICE)
    TOK = AutoTokenizer.from_pretrained(ENCODER_MODEL, use_fast=True)
    ENC = AutoModel.from_pretrained(ENCODER_MODEL).to(DEVICE).eval()
    MAXTOK = min(TOK.model_max_length, 512) - 2
    if DEVICE == "cpu":
        print("WARNING: no GPU available; the semantic metrics will be slow.")

## 3. Metric functions

**Lexical metrics.** WER and CER are computed on lowercase text without punctuation. In Punc-WER each
punctuation mark is a separate token (lowercase); PC-WER also distinguishes upper and lower case.

**Semantic metrics.** The reference is divided into fragments of 10-60 words (grouping complete sentences),
and each fragment is compared with the aligned part of the hypothesis (word alignment of jiwer). The score
of the video is the mean of the fragments weighted by their length, so long texts are not truncated.
SeMaScore follows the official algorithm (Sasindran et al., 2024; github.com/zenlab-edgeASR/SeMaScore), with
the multilingual encoder above.

In [ ]:
_HDR = re.compile(r"^[A-Za-zÁÉÍÓÚáéíóúñÑ][\w\- ]*:\s")

def read_header_and_text(filepath):
    # The ASR output files have a "Key: value" header, a blank line and the transcript.
    with open(filepath, "r", encoding="utf-8-sig") as f:
        ls = f.read().splitlines()
    start = 0
    while start < len(ls) and not ls[start].strip():
        start += 1
    end = start
    while end < len(ls) and end - start < 25 and _HDR.match(ls[end].strip()):
        end += 1
    head_end = end if (end > start and (end == len(ls) or not ls[end].strip())) else start
    meta = {}
    for l in ls[start:head_end]:
        k, _, v = l.partition(":"); meta[k.strip()] = v.strip()
    text = re.sub(r"\s+", " ", " ".join(l.strip() for l in ls[head_end:] if l.strip())).strip()
    return text, meta

def parse_asr_file(filepath):
    # Returns (transcript, processing time in seconds)
    text, meta = read_header_and_text(filepath)
    m = re.search(r"([\d\.]+)", meta.get("Tiempo transcurrido", ""))
    return text, (float(m.group(1)) if m else 0.0)

def clean_for_wer(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    return re.sub(r"\s+", " ", text).strip()

def clean_for_punc_wer(text):
    return " ".join(re.findall(r"\w+|[^\w\s]", text.lower()))

def clean_for_pc_wer(text):
    return " ".join(re.findall(r"\w+|[^\w\s]", text))

def term_recall(ref, hyp, terms=None):
    terms = TERMS if terms is None else terms
    tot = hit = 0
    for t in terms:
        pat = re.compile(r"\b" + re.escape(t) + r"\b", re.I)
        r_, h_ = len(pat.findall(ref)), len(pat.findall(hyp))
        tot += r_; hit += min(r_, h_)
    return hit / tot if tot else np.nan

In [ ]:
# ---- Semantic metrics on aligned fragments ------------------------------------------------------------
WIN_MIN_WORDS, WIN_MAX_WORDS = 10, 60

def raw_word_pairs(text):
    # List of (raw token index, raw token, clean word). Tokens without content (e.g., '-') are dropped.
    pairs = []
    for ti, tok in enumerate(text.split()):
        for sub in clean_for_wer(tok).split():
            pairs.append((ti, tok, sub))
    return pairs

def ref_windows(ref_raw):
    sents = [s for s in re.split(r"(?<=[.!?…])\s+", ref_raw) if s.strip()]
    wins, cur = [], []
    for s in sents:
        w = [c for _, _, c in raw_word_pairs(s)]
        while len(w) > WIN_MAX_WORDS:              # sentence longer than the maximum
            if cur: wins.append(cur); cur = []
            wins.append(w[:WIN_MAX_WORDS]); w = w[WIN_MAX_WORDS:]
        cur += w
        if len(cur) >= WIN_MIN_WORDS:
            wins.append(cur); cur = []
    if cur:
        if wins and len(cur) < WIN_MIN_WORDS: wins[-1] += cur
        else: wins.append(cur)
    return wins

def raw_join(pairs, idx):
    out, last = [], None
    for i in idx:
        ti, tok, _ = pairs[i]
        if ti != last: out.append(tok)
        last = ti
    return " ".join(out)

def aligned_windows(ref_raw, hyp_raw):
    rp, hp = raw_word_pairs(ref_raw), raw_word_pairs(hyp_raw)
    r_words, h_words = [c for _, _, c in rp], [c for _, _, c in hp]
    wins = ref_windows(ref_raw)
    bounds, k = [], 0
    for w in wins:
        bounds.append((k, k + len(w))); k += len(w)
    assert k == len(r_words), "The fragments do not cover the reference"
    owner = np.zeros(len(h_words), dtype=int)       # fragment that owns each hypothesis word
    if h_words:
        wo = jiwer.process_words(" ".join(r_words), " ".join(h_words))
        ref2win = np.zeros(len(r_words), dtype=int)
        for wi, (a, b) in enumerate(bounds): ref2win[a:b] = wi
        for ch in wo.alignments[0]:
            if ch.type in ("equal", "substitute"):
                for off in range(ch.hyp_end_idx - ch.hyp_start_idx):
                    owner[ch.hyp_start_idx + off] = ref2win[ch.ref_start_idx + off]
            elif ch.type == "insert":
                anchor = ref2win[ch.ref_start_idx - 1] if ch.ref_start_idx > 0 else 0
                owner[ch.hyp_start_idx:ch.hyp_end_idx] = anchor
    out = []
    for wi, (a, b) in enumerate(bounds):
        hidx = np.where(owner == wi)[0].tolist()
        out.append(dict(ref_raw=raw_join(rp, range(a, b)), hyp_raw=raw_join(hp, hidx) if hidx else "",
                        ref_clean=" ".join(r_words[a:b]), hyp_clean=" ".join(h_words[i] for i in hidx),
                        n_ref=b - a, n_hyp=len(hidx)))
    return out

def weighted_mean(vals, wins):
    w = np.array([max(x["n_ref"], x["n_hyp"]) for x in wins], float)
    v = np.array(vals, float); ok = ~np.isnan(v)
    return float(np.sum(v[ok] * w[ok]) / np.sum(w[ok])) if ok.any() else np.nan

def encode_tokens(text):
    # Contextual embeddings (layer ENCODER_LAYER) per token and character offsets.
    # Texts longer than MAXTOK tokens are encoded in word chunks.
    words, spans, pos = text.split(), [], 0
    for w in words:
        s = text.index(w, pos); spans.append((s, s + len(w))); pos = s + len(w)
    embs, offs, i = [], [], 0
    while i < len(words):
        j = len(words)
        while True:                                   # largest chunk that fits
            piece = text[spans[i][0]:spans[j - 1][1]]
            n = len(TOK(piece, add_special_tokens=False)["input_ids"])
            if n <= MAXTOK or j - i == 1: break
            j = i + max(1, (j - i) * MAXTOK // n - 1)
        base = spans[i][0]
        enc = TOK(piece, return_offsets_mapping=True, return_tensors="pt", truncation=True)
        om = enc.pop("offset_mapping")[0].tolist()
        with torch.no_grad():
            hs = ENC(**{k: v.to(DEVICE) for k, v in enc.items()},
                     output_hidden_states=True).hidden_states[ENCODER_LAYER][0].cpu()
        for t, (a, b) in enumerate(om):
            if b > a:                                  # skip special tokens
                embs.append(hs[t]); offs.append((a + base, b + base))
        i = j
    return (torch.stack(embs) if embs else torch.zeros(0, ENC.config.hidden_size)), offs

def segment_embeddings(text, segments, embs, offs):
    # Mean embedding of the tokens inside each segment (character positions).
    out, pos = [], 0
    for seg in segments:
        if not seg.strip():
            out.append(None); continue
        s = text.index(seg, pos); e = s + len(seg); pos = e
        idx = [k for k, (a, b) in enumerate(offs) if a < e and b > s]
        out.append(embs[idx].mean(0) if idx else None)
    return out

# ---- SeMaScore: character alignment and segmentation of the official implementation ----
def _edit_ops(s1, s2):
    # Replica of editDP/printChanges (official code): s1 = hypothesis, s2 = reference.
    # Aligned string: the character if it matches, '$' substitution, '-' deletion from s1, '+' addition from s2.
    n1, n2 = len(s1), len(s2)
    dp = [list(range(n2 + 1))] + [[i] + [0] * n2 for i in range(1, n1 + 1)]
    for i in range(1, n1 + 1):
        c, row, up = s1[i - 1], dp[i], dp[i - 1]
        for j in range(1, n2 + 1):
            row[j] = up[j-1] if s2[j-1] == c else 1 + min(row[j-1], up[j-1], up[j])
    x, i, j = [], n1, n2
    while i > 0 and j > 0:
        if s1[i-1] == s2[j-1]: x.append(s1[i-1]); i -= 1; j -= 1
        elif dp[i][j] == dp[i-1][j-1] + 1: x.append("$"); i -= 1; j -= 1
        elif dp[i][j] == dp[i-1][j] + 1: x.append("-"); i -= 1
        else: x.append("+"); j -= 1
    x += ["-"] * i + ["+"] * j
    return "".join(reversed(x))

def _split_mapped(text, aligned, skip):
    # sub_sentence_mapper1 (skip='-', text=reference) and sub_sentence_mapper2 (skip='+', text=hypothesis)
    segs, cur, c, k = [], "", 0, 0
    while c < len(text) and k < len(aligned):
        if text[c] == " " and aligned[k] == " ":
            segs.append(cur); cur = ""; c += 1; k += 1
        elif aligned[k] == skip:
            k += 1
        else:
            cur += text[c]; c += 1; k += 1
    cur += text[c:]
    segs.append(cur)
    return segs

def _one_minus_mer(a, b):
    if not a and not b: return 1.0
    al = _edit_ops(b, a)
    mism = sum(ch in "+-$" for ch in al)
    return 1 - mism / max(len(a), len(b))

def semascore_window(ref_clean, hyp_clean):
    if not ref_clean: return np.nan
    if not hyp_clean: return 0.0
    aligned = _edit_ops(hyp_clean, ref_clean)
    gseg = _split_mapped(ref_clean, aligned, "-")
    hseg = _split_mapped(hyp_clean, aligned, "+")
    if len(gseg) != len(hseg):
        return np.nan
    ge, go = encode_tokens(ref_clean); he, ho = encode_tokens(hyp_clean)
    gvec = segment_embeddings(ref_clean, gseg, ge, go)
    hvec = segment_embeddings(hyp_clean, hseg, he, ho)
    gmean = ge.mean(0)
    cos = lambda a, b: float(torch.nn.functional.cosine_similarity(a, b, dim=0))
    num = den = 0.0
    for gs, hs, gv, hv in zip(gseg, hseg, gvec, hvec):
        if gv is None: continue                      # empty reference segment
        imp = (cos(gv, gmean) + 1) / 2
        ss = (cos(gv, hv) + 1) / 2 if hv is not None else 0.5
        num += imp * ss * _one_minus_mer(gs, hs); den += imp
    return num / den if den else np.nan

def semantic_metrics(wins):
    refs_r = [w["ref_raw"] for w in wins]; hyps_r = [w["hyp_raw"] for w in wins]
    nonempty = [i for i, h in enumerate(hyps_r) if h.strip()]
    bs = np.zeros(len(wins))
    if nonempty:
        _, _, f1 = BS.score([hyps_r[i] for i in nonempty], [refs_r[i] for i in nonempty], batch_size=32)
        bs[nonempty] = f1.numpy()
    sb = np.zeros(len(wins))
    if nonempty:
        er = SBERT.encode([refs_r[i] for i in nonempty], convert_to_tensor=True)
        eh = SBERT.encode([hyps_r[i] for i in nonempty], convert_to_tensor=True)
        sb[nonempty] = st_util.cos_sim(er, eh).diagonal().cpu().numpy()
    sm = [semascore_window(w["ref_clean"], w["hyp_clean"]) for w in wins]
    return {"BERTScore": weighted_mean(bs, wins), "SeMaScore": weighted_mean(sm, wins),
            "SBERT-cos": weighted_mean(sb, wins),
            "Ventanas": len(wins), "Ventanas SeMa fallidas": int(np.isnan(sm).sum())}

In [ ]:
# ---- Speed-up of the semantic metrics (same results) ----
# (1) cache: each text (e.g., the reference fragments, identical for all models) is encoded once;
# (2) batches: the fragments of a hypothesis are encoded together in one forward pass.
BATCH_SIZE = 32
_ENC_CACHE = {}
_encode_tokens_original = encode_tokens

def encode_tokens(text):
    if text not in _ENC_CACHE:
        _ENC_CACHE[text] = _encode_tokens_original(text)
    return _ENC_CACHE[text]

def precode(texts):
    pending = [t for t in dict.fromkeys(texts) if t and t not in _ENC_CACHE]
    short = [t for t in pending if len(TOK(t, add_special_tokens=False)["input_ids"]) <= MAXTOK]
    for i in range(0, len(short), BATCH_SIZE):
        batch = short[i:i + BATCH_SIZE]
        enc = TOK(batch, return_offsets_mapping=True, return_tensors="pt", padding=True, truncation=True)
        om = enc.pop("offset_mapping")
        mask = enc["attention_mask"]
        with torch.no_grad():
            hs = ENC(**{k: v.to(DEVICE) for k, v in enc.items()},
                     output_hidden_states=True).hidden_states[ENCODER_LAYER].cpu()
        for b, t in enumerate(batch):
            embs, offs = [], []
            for k, (a, e) in enumerate(om[b].tolist()):
                if e > a and int(mask[b, k]) == 1:          # skip special tokens and padding
                    embs.append(hs[b, k]); offs.append((a, e))
            _ENC_CACHE[t] = (torch.stack(embs) if embs else torch.zeros(0, hs.shape[-1]), offs)
    for t in pending:                                    # long texts: chunked encoding
        if t not in _ENC_CACHE:
            _ENC_CACHE[t] = _encode_tokens_original(t)

def clear_cache():
    _ENC_CACHE.clear()
    if RECOMPUTE and torch.cuda.is_available():
        torch.cuda.empty_cache()

## 4. Per-video metrics of the nine configurations (raw output)

Partial results are saved after each model, so the computation can be resumed if the session is interrupted.
FH-Index is the Fernández-Huerta readability index of the complete text (textstat).
RTF = processing time / audio duration (both read from the header of the ASR output files).

In [ ]:
RESULTS_CSV = OUT_DIR / "results_per_video_raw.csv"

if RECOMPUTE:
    PARTIAL_CSV = OUT_DIR / "_partial_results.csv"
    RESTART = False      # True = ignore the partial results and recompute everything
    if PARTIAL_CSV.exists() and not RESTART:
        rows = pd.read_csv(PARTIAL_CSV).to_dict("records")
        done = {(r["Video / Contexto"], r["Modelo Whisper"]) for r in rows}
        print(f"Resuming: {len(done)} combinations already computed")
    else:
        rows, done = [], set()
    save_partial = lambda: pd.DataFrame(rows).to_csv(PARTIAL_CSV, index=False, encoding="utf-8-sig")

    t_total = time.time()
    for video, data in CONFIG.items():
        with open(data["referencia"], "r", encoding="utf-8") as f:
            ref_raw = re.sub(r"\s+", " ", f.read()).strip()
        ref_wer, ref_punc = clean_for_wer(ref_raw), clean_for_punc_wer(ref_raw)
        clear_cache()
        print(f"  {video}")
        if (video, REF_LABEL) not in done:
            rows.append({"Video / Contexto": video, "Modelo Whisper": REF_LABEL, "Parámetros": "-",
                         "Tiempo Inferencia (s)": 0.0, "CER (↓)": 0.0, "WER (↓)": 0.0, "Punc-WER (↓)": 0.0,
                         "PC-WER (↓)": 0.0, "BERTScore (↑)": 1.0, "SeMaScore (↑)": 1.0, "SBERT-cos (↑)": 1.0,
                         "F. Huerta (Legibilidad)": round(textstat.fernandez_huerta(ref_raw), 2)})
        for model, hyp_path in data["hipotesis"].items():
            if (video, model) in done:
                continue
            t0 = time.time()
            hyp_raw, proc_time = parse_asr_file(hyp_path)
            _, meta = read_header_and_text(hyp_path)
            md_ = re.search(r"([\d\.]+)", meta.get("Duración audio", ""))
            duration = float(md_.group(1)) if md_ else np.nan
            if not hyp_raw.strip():
                rows.append({"Video / Contexto": video, "Modelo Whisper": model, "Parámetros": PARAMS.get(model),
                             "Tiempo Inferencia (s)": proc_time, "CER (↓)": 1.0, "WER (↓)": 1.0,
                             "Punc-WER (↓)": 1.0, "PC-WER (↓)": 1.0, "BERTScore (↑)": 0.0, "SeMaScore (↑)": 0.0,
                             "F. Huerta (Legibilidad)": 0.0, "Archivo": os.path.basename(hyp_path)})
                save_partial(); continue
            hyp_wer = clean_for_wer(hyp_raw)
            wins = aligned_windows(ref_raw, hyp_raw)
            precode([w["ref_clean"] for w in wins] + [w["hyp_clean"] for w in wins])
            sem = semantic_metrics(wins)
            rows.append({
                "Video / Contexto": video, "Modelo Whisper": model, "Parámetros": PARAMS.get(model),
                "Tiempo Inferencia (s)": round(proc_time, 2),
                "CER (↓)": round(jiwer.cer(ref_wer, hyp_wer), 4),
                "WER (↓)": round(jiwer.wer(ref_wer, hyp_wer), 4),
                "Punc-WER (↓)": round(jiwer.wer(ref_punc, clean_for_punc_wer(hyp_raw)), 4),
                "PC-WER (↓)": round(jiwer.wer(clean_for_pc_wer(ref_raw), clean_for_pc_wer(hyp_raw)), 4),
                "BERTScore (↑)": round(sem["BERTScore"], 4),
                "SeMaScore (↑)": round(sem["SeMaScore"], 4),
                "SBERT-cos (↑)": round(sem["SBERT-cos"], 4),
                "F. Huerta (Legibilidad)": round(textstat.fernandez_huerta(hyp_raw), 2),
                "RTF": round(proc_time / duration, 4) if duration else np.nan,
                "Recall términos": term_recall(ref_raw, hyp_raw),
                "Ratio palabras": round(len(hyp_wer.split()) / max(len(ref_wer.split()), 1), 3),
                "Idioma": meta.get("Idioma", ""),
                "Ventanas SeMa fallidas": sem["Ventanas SeMa fallidas"],
                "Archivo": os.path.basename(hyp_path)})
            save_partial()
            print(f"     {model:15s} {time.time() - t0:5.1f} s  ({len(wins)} fragments)")
    df_final = pd.DataFrame(rows)
    df_final.to_csv(RESULTS_CSV, index=False, encoding="utf-8-sig")
    print(f"Finished in {(time.time() - t_total) / 60:.1f} min -> {RESULTS_CSV}")
else:
    df_final = pd.read_csv(FROZEN_DIR / "results_per_video_raw.csv")
    print("Loaded", FROZEN_DIR / "results_per_video_raw.csv")

COL = {"WER": "WER (↓)", "CER": "CER (↓)", "Punc-WER": "Punc-WER (↓)", "PC-WER": "PC-WER (↓)",
       "BERTScore": "BERTScore (↑)", "SeMaScore": "SeMaScore (↑)", "SBERT-cos": "SBERT-cos (↑)",
       "FH": "F. Huerta (Legibilidad)"}
HIGHER_BETTER = {"BERTScore", "SeMaScore", "SBERT-cos", "FH", "Recall términos"}
df_res = df_final[df_final["Modelo Whisper"] != REF_LABEL].copy()
df_res["video"] = df_res["Video / Contexto"].str.extract(r"(\d+)").astype(int)
df_res["model"] = df_res["Modelo Whisper"].astype(str)
for k, v in COL.items():
    if v in df_res: df_res[k] = df_res[v]
df_res = df_res.sort_values(["video", "model"]).reset_index(drop=True)
assert df_res.groupby("model").size().eq(15).all() and set(df_res.model) == set(MODEL_ORDER)
display(df_res.groupby("model")[list(COL) + ["RTF"]].mean().reindex(MODEL_ORDER).round(3))

In [ ]:
# Quality checks (full recomputation only)
if RECOMPUTE:
    leak = re.compile(r"(Dispositivo|Iteración|Modelo[- ](?:HF|Dispositivo)|Tiempo transcurrido|Palabras transcritas)\s*:")
    left = [(r.video, r.model) for r in df_res.itertuples()
            if leak.search(parse_asr_file(CONFIG[f"Video {r.video}"]["hipotesis"][r.model])[0])]
    print("Transcripts with header residues:", left or "none")
    print("Ratio of hypothesis/reference words > 1.3 (possible hallucination):")
    print(df_res.loc[df_res["Ratio palabras"] > 1.3, ["video", "model", "Ratio palabras", "WER"]].to_string(index=False) or "none")
    print("SeMaScore fragments that could not be aligned:", int(df_res["Ventanas SeMa fallidas"].sum()))
    r0 = re.sub(r"\s+", " ", open(CONFIG["Video 1"]["referencia"], encoding="utf-8").read()).strip()
    print("Sanity check (reference vs. itself):", {k: round(v, 4) for k, v in semantic_metrics(aligned_windows(r0, r0)).items()})

## 5. Subtitle metrics (UNE 153010:2012 constraints)

For each SRT file: number of subtitles, characters per second (CPS), display time and the percentage of
subtitles that meet the limits (at most 2 lines of 37 characters, at most 15 CPS, display time between 1 and 6 s).

In [ ]:
def _t2s(t):
    h, m, s = t.strip().replace(",", ".").split(":")
    return int(h) * 3600 + int(m) * 60 + float(s)

def read_srt(path):
    subs = []
    for blk in re.split(r"\n\s*\n", Path(path).read_text(encoding="utf-8-sig").strip()):
        L = blk.strip().splitlines()
        if len(L) < 3 or "-->" not in L[1]: continue
        a, b = (_t2s(x) for x in L[1].split("-->"))
        lines = [re.sub(r"<[^>]+>", "", l).strip() for l in L[2:]]
        subs.append((a, b, [l for l in lines if l]))
    return subs

def wrap_lines(text, width=MAX_CPL):
    lines, cur = [], ""
    for w in text.split():
        if not cur: cur = w
        elif len(cur) + 1 + len(w) <= width: cur += " " + w
        else: lines.append(cur); cur = w
    return lines + ([cur] if cur else [])

def srt_metrics(path):
    rows, prev_end = [], 0.0
    for a, b, lines in read_srt(path):
        text = " ".join(lines); d = b - a; n = len(text)
        rows.append(dict(chars=n, dur=d, cps=n / d if d > 0 else np.inf, lines=len(lines),
                         max_cpl=max((len(l) for l in lines), default=0), wrap=len(wrap_lines(text)),
                         overlap=a < prev_end - 1e-6, fh=textstat.fernandez_huerta(text) if text else np.nan))
        prev_end = b
    if not rows: return {}
    df = pd.DataFrame(rows); fin = df[np.isfinite(df.cps)]
    ok_len = df.wrap <= MAX_LINES; ok_cps = np.isfinite(df.cps) & (df.cps <= MAX_CPS)
    ok_dur = (df.dur >= MIN_DUR) & (df.dur <= MAX_DUR)
    pct = lambda s: 100 * s.mean()
    return {"Subtítulos": len(df), "Car/subt": df.chars.mean(), "Dur media (s)": df.dur.mean(),
            "CPS mediana": fin.cps.median(), "CPS p90": fin.cps.quantile(.9),
            "% CPL≤max (tal cual)": pct((df.max_cpl <= MAX_CPL) & (df.lines <= MAX_LINES)),
            "% cabe en líneas": pct(ok_len), "% CPS≤max": pct(ok_cps),
            "% duración en rango": pct(ok_dur), "% <min dur": pct(df.dur < MIN_DUR),
            "% >max dur": pct(df.dur > MAX_DUR), "% cumple long+CPS": pct(ok_len & ok_cps),
            "% cumple todo": pct(ok_len & ok_cps & ok_dur), "% solapados": pct(df.overlap),
            "FH-subt": df.fh.mean()}

if RECOMPUTE:
    rows = []
    for p in sorted(glob.glob(f"{ASR_DIR}/VIDEO-*/*.srt")):
        parts = os.path.basename(p)[:-4].split("__")
        if len(parts) != 3: continue
        alias, variant, seg = parts
        rows.append(dict(video=int(re.search(r"VIDEO-(\d+)", p).group(1)), model=ALIAS_TO_NAME.get(alias, alias),
                         variant=variant, seg=seg, **srt_metrics(p)))
    df_srt = pd.DataFrame(rows)
    df_srt.to_csv(OUT_DIR / "subtitle_metrics.csv", index=False, encoding="utf-8-sig")
else:
    df_srt = pd.read_csv(FROZEN_DIR / "subtitle_metrics.csv")

COLS_SRT = ["Subtítulos", "CPS p90", "% cabe en líneas", "% CPS≤max", "% duración en rango",
            "% cumple long+CPS", "% cumple todo", "FH-subt"]
display(df_srt[df_srt.variant == MAIN_VARIANT].groupby(["seg", "model"])[COLS_SRT].mean()
        .reindex([(s, m) for s in SEGMENTATIONS for m in MODEL_ORDER]).round(1))

## 6. Ablation of the post-processing components

* Terminology normalization (`norm`), initial prompt (`prompt`, only Small, Large-v3 and Large-v3-turbo) and,
  for Wav2Vec2, punctuation restoration (`punct`) against the raw output.
* Segmentation strategies (`rules`, `une`) against the native segmentation.

Wilcoxon signed-rank test (Pratt), Holm correction within each metric.

In [ ]:
def ablation(df, metrics, factor, levels, base):
    out = []
    for m in [x for x in MODEL_ORDER if x in set(df.model)]:
        d = df[df.model == m]; present = [l for l in levels if l in set(d[factor])]
        if base not in present or len(present) < 2: continue
        for c in metrics:
            P = d.pivot_table(index="video", columns=factor, values=c)
            for lv in present:
                if lv == base: continue
                x = P[[base, lv]].dropna(); dif = x[lv] - x[base]
                p = 1.0 if np.allclose(dif, 0) else wilcoxon(x[lv], x[base], zero_method="pratt").pvalue
                out.append(dict(Modelo=m, Métrica=c, Base=base, Variante=lv, media_base=x[base].mean(),
                                media_variante=x[lv].mean(), delta=dif.mean(), p_raw=p, n=len(x)))
    A = pd.DataFrame(out)
    if len(A): A["p_Holm"] = A.groupby("Métrica").p_raw.transform(lambda p: multipletests(p, method="holm")[1])
    return A

if RECOMPUTE:
    rows = []
    for var in VARIANTS:
        for video, data in build_config(var).items():
            ref = re.sub(r"\s+", " ", open(data["referencia"], encoding="utf-8").read()).strip()
            for model, path in data["hipotesis"].items():
                hyp, _ = parse_asr_file(path)
                rows.append(dict(video=int(re.search(r"(\d+)", video).group(1)), model=model, variant=var,
                                 WER=jiwer.wer(clean_for_wer(ref), clean_for_wer(hyp)),
                                 CER=jiwer.cer(clean_for_wer(ref), clean_for_wer(hyp)),
                                 **{"Punc-WER": jiwer.wer(clean_for_punc_wer(ref), clean_for_punc_wer(hyp)),
                                    "PC-WER": jiwer.wer(clean_for_pc_wer(ref), clean_for_pc_wer(hyp)),
                                    "Recall términos": term_recall(ref, hyp)}))
    df_abl = pd.DataFrame(rows)
    A_txt = ablation(df_abl, ["WER", "CER", "Punc-WER", "PC-WER", "Recall términos"], "variant", VARIANTS, "raw")
    A_seg = ablation(df_srt[df_srt.variant == MAIN_VARIANT], ["% cumple long+CPS", "% cumple todo", "CPS p90", "FH-subt"],
                     "seg", SEGMENTATIONS, "native")
    A_txt.to_csv(OUT_DIR / "ablation_postprocessing.csv", index=False, encoding="utf-8-sig")
    A_seg.to_csv(OUT_DIR / "ablation_segmentation.csv", index=False, encoding="utf-8-sig")
else:
    A_txt = pd.read_csv(FROZEN_DIR / "ablation_postprocessing.csv")
    A_seg = pd.read_csv(FROZEN_DIR / "ablation_segmentation.csv")
display(A_txt[A_txt["Métrica"] == "WER"].round(4))
display(A_seg[A_seg["Métrica"] == "% cumple long+CPS"].round(4))

## 7. Punctuation restoration for Wav2Vec 2.0

(a) Wav2Vec2 without and with restoration (Wav2Vec2+PR); (b) Wav2Vec2+PR against each Whisper model with its
native punctuation. FH-subt and compliance are computed on the UNE-based subtitles.
Requires the `wav2vec2__punct` files created by `scripts/punctuation_restoration_wav2vec.py`.

In [ ]:
W2V_PR = "Wav2Vec2+PR"
if RECOMPUTE and glob.glob(f"{ASR_DIR}/VIDEO-*/wav2vec2__punct.txt"):
    rows = []
    for var in ["raw", "punct"]:
        for video, data in build_config(var).items():
            ref = re.sub(r"\s+", " ", open(data["referencia"], encoding="utf-8").read()).strip()
            for model, path in data["hipotesis"].items():
                if var == "punct" and model != "Wav2Vec2":
                    continue
                hyp, _ = parse_asr_file(path)
                rows.append(dict(video=int(re.search(r"(\d+)", video).group(1)),
                                 model=W2V_PR if var == "punct" else model, variant=var,
                                 **{"WER": jiwer.wer(clean_for_wer(ref), clean_for_wer(hyp)),
                                    "Punc-WER": jiwer.wer(clean_for_punc_wer(ref), clean_for_punc_wer(hyp)),
                                    "PC-WER": jiwer.wer(clean_for_pc_wer(ref), clean_for_pc_wer(hyp)),
                                    "FH": textstat.fernandez_huerta(hyp)}))
    df_p = pd.DataFrame(rows)
    s = df_srt[(df_srt.seg == "une") & ((df_srt.variant == "raw") |
               ((df_srt.variant == "punct") & (df_srt.model == "Wav2Vec2")))].copy()
    s.loc[s.variant == "punct", "model"] = W2V_PR
    df_p = df_p.merge(s[["video", "model", "variant", "FH-subt", "% cumple long+CPS"]],
                      on=["video", "model", "variant"], how="left")
    met = ["WER", "Punc-WER", "PC-WER", "FH", "FH-subt", "% cumple long+CPS"]
    order = list(MODEL_ORDER); order.insert(order.index("Wav2Vec2") + 1, W2V_PR)
    means = df_p.groupby("model")[met].mean().reindex(order)

    # (a) effect of the restoration on Wav2Vec2
    P = df_p[df_p.model.isin(["Wav2Vec2", W2V_PR])].pivot_table(index="video", columns="model", values=met)
    fil = []
    for c in met:
        x = P[c][["Wav2Vec2", W2V_PR]].dropna(); d = x[W2V_PR] - x["Wav2Vec2"]
        p = 1.0 if np.allclose(d, 0) else wilcoxon(x[W2V_PR], x["Wav2Vec2"], zero_method="pratt").pvalue
        fil.append(dict(Métrica=c, Wav2Vec2=x["Wav2Vec2"].mean(), Wav2Vec2_PR=x[W2V_PR].mean(), delta=d.mean(), p_raw=p))
    effect = pd.DataFrame(fil); effect["p_Holm"] = multipletests(effect.p_raw, method="holm")[1]

    # (b) Wav2Vec2+PR against each Whisper model
    whisper = [m for m in MODEL_ORDER if m != "Wav2Vec2"]
    fil = []
    for c in met:
        Q = df_p[df_p.model.isin(whisper + [W2V_PR])].pivot(index="video", columns="model", values=c).dropna()
        n = len(Q); fr = friedmanchisquare(*[Q[m] for m in Q]).pvalue
        for m in whisper:
            d = Q[W2V_PR] - Q[m]
            if np.allclose(d, 0): p, r = 1.0, 0.0
            else:
                p = wilcoxon(Q[W2V_PR], Q[m], zero_method="pratt").pvalue
                za = wilcoxon(Q[W2V_PR], Q[m], zero_method="pratt", method="approx")
                r = abs(getattr(za, "zstatistic", norm.isf(za.pvalue / 2))) / math.sqrt(n)
            fil.append(dict(Métrica=c, Friedman_p=fr, Comparado=m, media_W2V_PR=Q[W2V_PR].mean(),
                            media_modelo=Q[m].mean(), p_raw=p, r=r))
    vs_whisper = pd.DataFrame(fil)
    vs_whisper["p_Holm"] = vs_whisper.groupby("Métrica").p_raw.transform(lambda p: multipletests(p, method="holm")[1])
    vs_whisper["Significativo"] = vs_whisper.p_Holm < 0.05
    means.to_csv(OUT_DIR / "punctuation_restoration_means.csv", encoding="utf-8-sig")
    effect.to_csv(OUT_DIR / "punctuation_restoration_effect.csv", index=False, encoding="utf-8-sig")
    vs_whisper.to_csv(OUT_DIR / "punctuation_restoration_vs_whisper.csv", index=False, encoding="utf-8-sig")
else:
    means = pd.read_csv(FROZEN_DIR / "punctuation_restoration_means.csv", index_col=0)
    effect = pd.read_csv(FROZEN_DIR / "punctuation_restoration_effect.csv")
    vs_whisper = pd.read_csv(FROZEN_DIR / "punctuation_restoration_vs_whisper.csv")
display(means.round(3)); display(effect.round(4))

## 8. Statistical analysis

* Friedman test for each metric (15 videos, 9 configurations) with Kendall's $W = \chi^2_F / (n(k-1))$.
* Pairwise Wilcoxon signed-rank tests (Pratt) with Holm correction within each metric; effect size
  $r = |Z|/\sqrt{N}$, with $Z$ from the normal approximation.
* Descriptive statistics with 95% bootstrap confidence intervals of the mean (10,000 resamples, seed 42).
* Sensitivity analysis: Friedman test and Kendall's $W$ after excluding the videos where any model obtains
  WER < 0.02 (possible bias of the references, which were obtained by post-editing an ASR draft).

In [ ]:
METRICS = ["CER", "WER", "Punc-WER", "PC-WER", "BERTScore", "SeMaScore", "FH"]

def friedman_w(P):
    stat, p = friedmanchisquare(*[P[m].values for m in P.columns])
    n, k = P.shape
    return stat, p, stat / (n * (k - 1))

friedman_rows, pair_rows = [], []
for c in METRICS:
    P = df_res.pivot(index="video", columns="model", values=c)[MODEL_ORDER].dropna()
    stat, p, W = friedman_w(P)
    friedman_rows.append(dict(Metric=COL[c], n=len(P), k=P.shape[1], chi2=stat, p=p, W=W))
    comps, pvals, rs = [], [], []
    for a, b in itertools.combinations(MODEL_ORDER, 2):
        if np.allclose(P[a] - P[b], 0):
            pv, r = 1.0, 0.0
        else:
            pv = wilcoxon(P[a], P[b], zero_method="pratt").pvalue
            za = wilcoxon(P[a], P[b], zero_method="pratt", method="approx")
            r = abs(getattr(za, "zstatistic", norm.isf(za.pvalue / 2))) / math.sqrt(len(P))
        comps.append((a, b)); pvals.append(pv); rs.append(r)
    reject, p_holm, _, _ = multipletests(pvals, method="holm")
    for (a, b), pr, ph, rej, r in zip(comps, pvals, p_holm, reject, rs):
        pair_rows.append(dict(Metric=COL[c], A=a, B=b, p_raw=pr, p_Holm=ph, r=r, significant=bool(rej)))

df_friedman = pd.DataFrame(friedman_rows)
df_pairs = pd.DataFrame(pair_rows)
df_friedman.to_csv(OUT_DIR / "stats" / "friedman.csv", index=False, encoding="utf-8-sig")
df_pairs.to_csv(OUT_DIR / "stats" / "wilcoxon_holm.csv", index=False, encoding="utf-8-sig")
display(df_friedman.round(4))
print("Significant pairs after Holm correction:", int(df_pairs.significant.sum()), "of", len(df_pairs))
display(df_pairs.groupby("Metric").significant.sum())

In [ ]:
# Descriptive statistics with 95% bootstrap confidence intervals (one generator, fixed order of metrics and models)
rng = np.random.default_rng(SEED)
desc_rows = []
for c in METRICS:
    for m in MODEL_ORDER:
        x = df_res[df_res.model == m].sort_values("video")[c].values
        boot = rng.choice(x, (N_BOOT, len(x))).mean(1)
        lo, hi = np.quantile(boot, [0.025, 0.975]); q1, q3 = np.quantile(x, [0.25, 0.75])
        desc_rows.append(dict(Metric=c, Model=m, Mean=x.mean(), Std=x.std(ddof=1), Median=np.median(x),
                              IQR=q3 - q1, CI95_low=lo, CI95_high=hi))
df_desc = pd.DataFrame(desc_rows)
df_desc.to_csv(OUT_DIR / "stats" / "descriptive.csv", index=False, encoding="utf-8-sig")
display(df_desc[df_desc.Metric == "WER"].round(4))

In [ ]:
# Sensitivity analysis: possible bias of the references towards Whisper
excluded = sorted(df_res.groupby("video").WER.min().loc[lambda s: s < SENSITIVITY_WER].index)
print("Excluded videos (any model with WER <", SENSITIVITY_WER, "):", excluded)
rows = []
for name, d in [("todos", df_res), (f"sin WER<{SENSITIVITY_WER}", df_res[~df_res.video.isin(excluded)])]:
    for c in METRICS:
        P = d.pivot(index="video", columns="model", values=c)[MODEL_ORDER].dropna()
        stat, p, W = friedman_w(P)
        ranks = P.rank(axis=1, ascending=c not in HIGHER_BETTER).mean()
        rows.append({"Conjunto": name, "Métrica": c, "n": len(P), "Friedman p": p, "W": W, **ranks.round(2).to_dict()})
df_sens = pd.DataFrame(rows)
df_sens.to_csv(OUT_DIR / "stats" / "sensitivity.csv", index=False, encoding="utf-8-sig")
display(df_sens)

## 9. Figures and tables of the paper

The figures (Figs. 3-5) and the LaTeX tables are generated by the scripts from the CSV files.

In [ ]:
src = OUT_DIR if RECOMPUTE else FROZEN_DIR
for script, args in [("make_figures.py", [str(src), str(OUT_DIR / "figures")]),
                     ("make_tables.py", [str(src), str(OUT_DIR / "tables")])]:
    r = subprocess.run([sys.executable, str(REPO_DIR / "scripts" / script), *args], capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])